# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [11]:
#!pip install langchain_community langchain_experimental langchain_openai

In [12]:
from typing import Optional

from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [13]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain_classic.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [14]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [15]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [16]:
from langchain_classic.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain_classic.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
     "You are a tech research planner who specializes in analyzing hardware releases and innovations. "
    "Create a structured research plan to investigate the following objective: {objective}. "
    "Break it down into specific, actionable research tasks that will gather comprehensive information."
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper(serpapi_api_key=SERPAPI_API_KEY)

# Custom analysis tool
def analyze_specs(input_text):
    """Analyzes and structures technical specifications"""
    analysis_prompt = PromptTemplate.from_template(
        "You are a hardware analyst. Given this information about graphics cards: {input_text}, "
        "extract and organize key specifications including: GPU model, memory size, performance metrics, "
        "price range, and release date. Present in a clear, structured format."
    )
    analysis_chain = LLMChain(llm=OpenAI(temperature=0), prompt=analysis_prompt)
    return analysis_chain.run(input_text=input_text)

tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for finding current information about new graphics card releases, specifications, benchmarks, and reviews. Use this to search for latest GPU news and technical details.",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for creating a research plan to investigate graphics cards. Input: an objective to create a research plan for. Output: a structured list of research tasks.",
    ),
    Tool(
        name="SpecAnalyzer",
        func=analyze_specs,
        description="useful for analyzing and organizing technical specifications of graphics cards. Input: raw information about GPU specs. Output: structured analysis of key specifications.",
    ),
]


prefix = """You are a tech research AI assistant specializing in computer hardware analysis. Your objective is: {objective}. 

You should gather information systematically, compare specifications, and create comprehensive summaries. 
Take into account these previously completed tasks: {context}."""

suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [17]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [18]:
OBJECTIVE = "Research the latest graphics cards released in 2025 and create a comprehensive summary report with key specifications, performance improvements, and pricing"

In [19]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [20]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...
Thought: To create a comprehensive summary report, I need to gather information systematically and compare specifications of the latest graphics cards released in 2025.
Action: TODO
Action Input: "Create a research plan for investigating the latest graphics cards released in 2025."Thought: To create a comprehensive summary report, I need to gather information systematically and compare specifications of the latest graphics cards released in 2025.
Action: TODO
Action Input: "Create a research plan for investigating the latest graphics cards released in 2025."
Observation: 

Objective: To investigate the latest graphics cards released in 2025 and analyze their features, performance, and impact on the tech industry.

Research Plan:

1. Identify the top graphics card manufacturers in 2025: The first step is to identify the leadi

/tmp/ipykernel_1662/386726618.py:23: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  return analysis_chain.run(input_text=input_text)



Observation: 

GPU Model: This refers to the specific model of the graphics processing unit (GPU) used in the graphics card.

Memory Size: This refers to the amount of memory (in gigabytes) available on the graphics card for storing and processing data.

Performance Metrics: This includes key performance metrics such as clock speed, memory bandwidth, and number of cores, which determine the overall performance of the graphics card.

Price Range: This refers to the range of prices at which the graphics card is available, typically based on the different versions or configurations of the card.

Release Date: This refers to the date when the graphics card was first released to the market.

Example:

GPU Model: NVIDIA GeForce RTX 2080 Ti

Memory Size: 11 GB GDDR6

Performance Metrics: 1350 MHz base clock speed, 616 GB/s memory bandwidth, 4352 CUDA cores

Price Range: $999 - $1,199

Release Date: September 2018
Thought: I should also use the Search function to gather current information an

{'objective': 'Research the latest graphics cards released in 2025 and create a comprehensive summary report with key specifications, performance improvements, and pricing'}